# Notebook for Library merging

In [1]:
# Auto-reload edited modules (Benchmarking/, Models/, ...) before each cell runs,
# so editing a .py file takes effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
from sklearn.model_selection import ParameterGrid
from Models.config_parser import load_config, load_dataset_config
from Models.dataset_and_models import Dataset, Model

In [3]:
# Load configurations
CONFIG = "../configs/config.yaml"

model_config   = load_config(CONFIG)        # {model_key: {param: [values]}}
dataset_config = load_dataset_config(CONFIG) # {dataset_key: {n_features: [...], n_samples: [...]}}

In [4]:
# All model × hyperparameter combinations
model_runs = [
    (model_key, params)
    for model_key, param_grid in model_config.items()
    for params in ParameterGrid(param_grid)
]

# All dataset × (n_features, n_samples) combinations
dataset_runs = [
    (dataset_key, params)
    for dataset_key, param_grid in dataset_config.items()
    for params in ParameterGrid(param_grid)
]


In [5]:
import yaml

# Each (model, dataset) cell is one runner.run() call. Within a cell the runner executes
# one exact oracle plus every selected approximation spec (library × approximator × budget),
# writing one results.csv row per backend run. So the real total — the number you can
# compare against len(results) in the reader cell below — multiplies the cells by the
# per-cell backend count, not just the cells themselves.
n_cells = len(model_runs) * len(dataset_runs)

with open(CONFIG) as f:
    _bench = yaml.safe_load(f)["benchmark"]
n_specs = len(_bench["libraries"]) * len(_bench["approximators"]) * len(_bench["budgets"])
runs_per_cell = 1 + n_specs  # 1 exact oracle + approximation sweep
total_runs = n_cells * runs_per_cell

print(f"{len(model_runs)} model configs × {len(dataset_runs)} dataset configs = {n_cells} (model, dataset) cells")
print(
    f"per cell: 1 oracle + {n_specs} approx specs "
    f"({len(_bench['libraries'])} libs × {len(_bench['approximators'])} approximators "
    f"× {len(_bench['budgets'])} budgets) = {runs_per_cell} backend runs"
)
print(f"→ {n_cells} × {runs_per_cell} = {total_runs} total backend runs / results.csv rows")

4 model configs × 6 dataset configs = 24 (model, dataset) cells
per cell: 1 oracle + 6 approx specs (3 libs × 2 approximators × 1 budgets) = 7 backend runs
→ 24 × 7 = 168 total backend runs / results.csv rows


### Iterate trough every combination and train the model for explanation

In [7]:
import yaml
from tqdm import tqdm
from Benchmarking import BenchmarkRunner
from Benchmarking.backends import (
    ShapTrueValueBackend,
    ShapApproxBackend,
    ShapIQApproxBackend,
    LightShapApproxBackend,
)

# Map config library names to their approximation backend classes.
APPROX_BACKEND_BY_LIBRARY = {
    "shap": ShapApproxBackend,
    "shapiq": ShapIQApproxBackend,
    "lightshap": LightShapApproxBackend,
}

# All benchmark knobs come from the config's `benchmark` section (loaded the same
# way as the model/dataset configs). No defaults: every knob must be set there.
with open(CONFIG) as f:
    benchmark_config = yaml.safe_load(f)["benchmark"]
n_background = benchmark_config["n_background"]
n_eval = benchmark_config["n_eval"]
BUDGETS = benchmark_config["budgets"]
APPROXIMATORS = benchmark_config["approximators"]
APPROX_BACKENDS = [APPROX_BACKEND_BY_LIBRARY[library] for library in benchmark_config["libraries"]]

# Exact ground truth: shap.Explainer auto-detects TreeSHAP / LinearSHAP — polynomial,
# exact, and (with background data) interventional == the marginal value function that
# every approximator below targets. One cheap oracle per cell, recomputed each run.
TRUE_VALUE_BACKENDS = [ShapTrueValueBackend]

# Approximation sweep: only the (library × approximator × budget) combinations
# selected in the config. The nominal budget differs in units across libraries, so
# n_model_evals — the real model-call count recorded by the runner — is the
# comparable budget axis.
approximation_specs = [
    (backend, {"approximator": approximator, "budget": budget})
    for backend in APPROX_BACKENDS
    for approximator in APPROXIMATORS
    for budget in BUDGETS
]

runner = BenchmarkRunner(
    true_value_backends=TRUE_VALUE_BACKENDS,
    approximation_specs=approximation_specs,
    output_csv="../Benchmarking/results.csv",
    n_background=n_background,
    n_eval=n_eval,
)

# Progress bar weighted by n_features rather than cell count. Per cell the work is
# n_eval × n_background × (#specs) × evals_per_explanation, and only the last factor
# varies across cells — it grows with n_features (permutation uses ~2·n_features evals
# for wide data, and per-row prediction cost rises with width too). So a 5000-feature
# gisette cell dwarfs a 4-feature one; weighting the bar by n_features tracks real
# compute far better than "cells completed / total cells" (which would race through the
# cheap cells then appear to stall on the expensive ones).
total_weight = sum(
    dataset_params.get("n_features", 1)
    for _, dataset_params in dataset_runs
    for _ in model_runs
)

with tqdm(total=total_weight, desc="benchmark", unit="feat") as bar:
    for dataset_key, dataset_params in dataset_runs:
        dataset_enum = Dataset[dataset_key.upper()]
        ds = dataset_enum.load_dataset(**dataset_params)
        X, y = ds["X"], ds["y"]
        weight = dataset_params.get("n_features", 1)

        for model_key, model_params in model_runs:
            bar.set_postfix_str(f"{dataset_key} nf={dataset_params.get('n_features')} | {model_key}")
            trainer = Model[model_key.upper()].get_model_with_params(dataset_enum, model_params)
            trainer.fit(X, y, task=ds["task"])

            runner.run(
                model=trainer.get_model(),
                X=X,
                run_meta={
                    "dataset": dataset_key,
                    "model": model_key,
                    "n_features": dataset_params.get("n_features"),
                    "n_samples": dataset_params.get("n_samples"),
                },
            )
            bar.update(weight)

print("Done. Results written to ../Benchmarking/results.csv")

benchmark:   0%|          | 0/1384 [00:00<?, ?feat/s, ames_housing nf=4 | random_forest]c:\Users\timli\OneDrive\Desktop\Master\TTML\PR_ModeAgnostic\.venv\Lib\site-packages\shapiq\imputer\marginal_imputer.py:109: UserWarning: The sample size is larger than the number of data points in the background set. Reducing the sample size to the number of background samples.
  self.init_background(self.data)
c:\Users\timli\OneDrive\Desktop\Master\TTML\PR_ModeAgnostic\.venv\Lib\site-packages\shapiq\approximator\regression\base.py:160: UserWarning: Not all budget is required due to the border-trick.
  self._sampler.sample(budget)
c:\Users\timli\OneDrive\Desktop\Master\TTML\PR_ModeAgnostic\.venv\Lib\site-packages\shapiq\approximator\regression\base.py:160: UserWarning: Not all budget is required due to the border-trick.
  self._sampler.sample(budget)
c:\Users\timli\OneDrive\Desktop\Master\TTML\PR_ModeAgnostic\.venv\Lib\site-packages\shapiq\approximator\regression\base.py:160: UserWarning: Not all bu

Done. Results written to ../Benchmarking/results.csv


In [8]:
import pandas as pd

results = pd.read_csv("../Benchmarking/results.csv")
print(f"{len(results)} rows written")

cols = [
    "dataset", "model", "n_features", "backend", "approximator", "budget",
    "n_model_evals", "runtime_s", "mean_abs_diff", "relative_mae",
    "sign_agreement", "mean_sample_rho",
]
results[cols].head(20)

168 rows written


,dataset,model,n_features,backend,approximator,budget,n_model_evals,runtime_s,mean_abs_diff,relative_mae,sign_agreement,mean_sample_rho
0,ames_housing,random_forest,4,shap_true_value,NaN,NaN,NaN,0.0020,NaN,NaN,NaN,NaN
1,ames_housing,random_forest,4,shap_approx,kernel,64.0,35100.0,0.1801,5.507594e-04,4.337937e-08,1.0,1.000
2,ames_housing,random_forest,4,shap_approx,permutation,64.0,132167.0,0.6683,5.507594e-04,4.337937e-08,1.0,1.000
3,ames_housing,random_forest,4,shapiq_approx,kernel,64.0,40051.0,2.7390,7.036062e-04,5.541802e-08,1.0,1.000
4,ames_housing,random_forest,4,shapiq_approx,permutation,64.0,155051.0,14.8668,2.906872e+02,2.289535e-02,1.0,0.996
5,ames_housing,random_forest,4,lightshap_approx,kernel,64.0,30100.0,0.1796,5.507594e-04,4.337937e-08,1.0,1.000
6,ames_housing,random_forest,4,lightshap_approx,permutation,64.0,30100.0,0.2362,5.507594e-04,4.337937e-08,1.0,1.000
7,ames_housing,decision_tree,4,shap_true_value,NaN,NaN,NaN,0.0009,NaN,NaN,NaN,NaN
8,ames_housing,decision_tree,4,shap_approx,kernel,64.0,35100.0,0.1410,1.232384e-03,9.635819e-08,1.0,1.000
9,ames_housing,decision_tree,4,shap_approx,permutation,64.0,132155.0,0.3469,1.232383e-03,9.635819e-08,1.0,1.000


## Ground-truth validation (run once)

The exact reference used across the whole sweep is `shap`'s auto-detected **polynomial**
explainer (TreeSHAP / LinearSHAP) — fast at any feature count. To confirm this cheap oracle
really equals the exact Shapley value, compare it on a small problem (california-housing,
8 features) against an **independent brute-force exact**: `shapiq`'s general model-agnostic
explainer at the maximum budget `2⁸ = 256` (full coalition enumeration) — a different
library *and* a different algorithm. Agreement to numerical precision validates the oracle.
One-off check, **not** part of the main sweep.

In [9]:
from Benchmarking.backends import ShapTrueValueBackend, ShapIQTrueValueBackend

val_ds = Dataset.CALIFORNIA_HOUSING.load_dataset(n_samples=300, n_features=8)
Xv, yv = val_ds["X"], val_ds["y"]
val_trainer = Model.RANDOM_FOREST.get_model_with_params(
    Dataset.CALIFORNIA_HOUSING, {"n_estimators": 50, "max_depth": 6}
)
val_trainer.fit(Xv, yv, task=val_ds["task"])
val_model = val_trainer.get_model()

bg, xe = Xv.iloc[:100], Xv.iloc[100:120]

# Oracle: polynomial TreeSHAP (the reference used across the whole sweep)
poly = ShapTrueValueBackend(val_model, bg).run_explainer(xe)

# Independent brute-force exact: shapiq's general model-agnostic explainer at the maximum
# budget = 2^n_features (full coalition enumeration) — different library, different algorithm.
shapiq_exact = ShapIQTrueValueBackend(val_model, bg).run_explainer(xe)

print("max |TreeSHAP - shapiq exact (full budget)|:", float((poly - shapiq_exact).abs().values.max()))
print("-> agree to numerical precision; oracle validated.")

max |TreeSHAP - shapiq exact (full budget)|: 0.0002920721205778576
-> agree to numerical precision; oracle validated.
